In [ ]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.append('..')
import collections
import os

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Subset

from src.dataset import make_loaders, simulate_imbalance
from src.evaluate import predict_loader, summarize
from src.experiment import run_experiment, save_result
from src.model import CNNBaseline
from src.sampler import BalancedBatchSampler
from src.train import fit, make_scheduler
from src.utils import get_device, set_seed

SEED, EPOCHS, IMG_SIZE, BATCH = 42, 30, 128, 32
DATA = '../data/processed'
RESULTS = '../results/ablations.csv'
device = get_device()



train_ds, _, train_loader, val_loader = make_loaders(DATA, IMG_SIZE, BATCH, SEED)
print(train_ds.class_to_idx)

keep_fraction = {0: 0.2, 4: 0.3}   
keep = simulate_imbalance(train_ds.targets, keep_fraction, SEED)
imb_ds = Subset(train_ds, keep)
imb_targets = [train_ds.targets[i] for i in keep]
print(pd.Series(imb_targets).value_counts().sort_index())

pd.DataFrame({'index': keep, 'path': [train_ds.samples[i][0] for i in keep]}) \
  .to_csv('../data/processed/imbalanced_train_indices.csv', index=False)

paths = {
    'data': '../data/processed',
    'ckpt_dir': '../checkpoints',
    'log_dir': '../logs',
    'results': '../results/experiments.csv',
}

In [ ]:
sampler = BalancedBatchSampler(imb_targets , 32 , num_batches=10 , seed=SEED)
batch=next(iter(sampler))
print(collections.Counter(imb_targets[i] for i in batch))

In [ ]:
first_row = run_experiment('imb_standard', device , paths ,  train_subset=keep, sampler_type='none')
second_row = run_experiment('imb_balanced', device , paths, train_subset=keep, sampler_type='balanced')
print(first_row)
print(second_row)

In [ ]:
paths_b = {**paths, 'results': '../results/balanced.csv'}
run_experiment('imb_shuffle', device, paths_b, epochs=20,
               imbalance=keep_fraction, balanced=False)
run_experiment('imb_balanced', device, paths_b, epochs=20,
               imbalance=keep_fraction, balanced=True)

In [ ]:
run_experiment('baseline', device, paths, epochs=20)

In [ ]:
exp = pd.read_csv('../results/experiments.csv')

keep = ['baseline', 'imb_standard', 'imb_balanced']
res = exp[exp['experiment'].isin(keep)]

cols = ['experiment', 'accuracy', 'macro_f1',
        'recall_ambulance', 'recall_minibus']
print(res[cols].round(3).to_string(index=False))

In [ ]:
run_experiment('aug', device, paths, augment=True, epochs=20)

In [ ]:
exp = pd.read_csv('../results/experiments.csv')

keep = ['baseline', 'imb_standard', 'imb_balanced']
res = exp[exp['experiment'].isin(keep)]

cols = ['experiment', 'accuracy', 'macro_f1',
        'recall_ambulance', 'recall_minibus']
print(res[cols].round(3).to_string(index=False))

In [ ]:
run_experiment('imb_shuffle_v2', device, paths_b, epochs=20,
               imbalance=keep_fraction, balanced=False)



In [ ]:
run_experiment('imb_balanced_v2', device, paths_b, epochs=20,
               imbalance=keep_fraction, balanced=True)